# Framework 2/4 — Causal Discovery & Graph Validation

> **Thesis:** Toward Actionable Root Cause Analysis: Integrating Causal Discovery and
> Counterfactual XAI for Predictive Maintenance
>
> **Prototype dataset:** NASA C-MAPSS FD001 · **Notebook 2 of 4** · 2026-07-30

The framework has nine modules. This notebook covers **Module 3 (causal discovery with PCMCI+) and Module 4 (graph validation)**.

All settings come from `pipeline_config.py` in the repo root — change values there, not in the
notebook. All logic lives in `src/causalpdm/framework/`, so the notebook stays a narrative over
tested code rather than a place where logic hides.

## Setup

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

import pipeline_config as cfg
from causalpdm.framework import artifacts

from causalpdm.framework import discovery, validation

prep = artifacts.load("prep")
feat = artifacts.load("feat")

variables = ([v for v in cfg.CAUSAL_VARIABLES if v in prep["feature_cols"]]
             if cfg.CAUSAL_VARIABLES else prep["feature_cols"])
print(f"Loaded notebook 1 results · {len(variables)} causal variables")

Loaded notebook 1 results · 11 causal variables


## Module 3 — Causal discovery with PCMCI+

### Why PCMCI+

PCMCI+ (Runge, 2020) is built for exactly this data shape: many sensors, strong autocorrelation,
and causes that act both instantly and after a delay. It works in two phases — first it narrows
down candidate parents for each variable, then it tests each surviving link while conditioning on
the parents of *both* endpoints, which is what controls false positives in autocorrelated series.

### Three assumptions we are making, stated openly

A thesis has to defend these, so they are worth naming:

1. **Causal sufficiency** — no hidden common causes. Degradation *is* one, which is exactly why we
   detrended in Module 1.
2. **Faithfulness** — no variable is a deterministic function of others. This is why the health
   indicator was kept out.
3. **Stationarity** — the mechanism is stable over an engine's life. Detrending helps here too.

### Pooling engines correctly

We have 100 short trajectories, not one long one. Concatenating them end to end would invent
causal links across engine boundaries (the last cycle of engine 3 "causing" the first cycle of
engine 4). Instead each engine is passed as a separate *realisation* of the same mechanism, which
is both correct and — measured — faster than concatenating.

In [2]:
engines = sorted(prep["detrended"]["unit"].unique())[: cfg.ENGINES_FOR_DISCOVERY]
discovery_data = prep["detrended"][prep["detrended"]["unit"].isin(engines)]

print(f"Engines pooled : {len(engines)}")
print(f"Rows           : {len(discovery_data):,}")
print(f"Max lag        : {cfg.MAX_LAG} cycles")
print(f"Significance   : {cfg.SIGNIFICANCE}")
print(f"Search bounds  : max_conds_dim={cfg.MAX_CONDITION_SET_SIZE}, max_combinations={cfg.MAX_COMBINATIONS}")

Engines pooled : 10
Rows           : 2,136
Max lag        : 2 cycles
Significance   : 0.01
Search bounds  : max_conds_dim=3, max_combinations=2


### A practical finding worth reporting in the thesis

Those last two settings are not cosmetic tuning — they are what makes the algorithm finish at all.

Measured on this data: with 10 engines, discovery takes about 11 seconds. With 20 engines and no
search bounds, it did not finish in 9 minutes. The reason is counterintuitive and worth stating
plainly: **more data made it slower.** Doubling the data doubles statistical power, so weak links
that were previously rejected now survive, the candidate graph stays dense, and the number of
conditioning sets to check grows combinatorially.

The practical lesson — *constraint-based causal discovery does not scale naively to industrial
sensor data, because sensor redundancy keeps the graph dense* — is a genuine methodological
result, not an implementation complaint.

In [3]:
disc = artifacts.cached("disc", lambda: discovery.run(
    discovery_data, variables,
    tau_max=cfg.MAX_LAG,
    pc_alpha=cfg.SIGNIFICANCE,
    max_conds_dim=cfg.MAX_CONDITION_SET_SIZE,
    max_combinations=cfg.MAX_COMBINATIONS,
))

print(f"Directed causal links found: {disc['n_directed_edges']}")
print("\nAll link types PCMCI+ reported:")
print(disc["link_types"].to_string())

Directed causal links found: 16

All link types PCMCI+ reported:
o-o    24
-->    16
<--     2


Note that PCMCI+ reports more than just `-->`. Link types like `o-o` mean *there is a dependency
but its direction could not be determined from the data*. Those are **not** counted as causal
edges here — treating them as directed would overstate what was actually discovered.

In [4]:
print("Strongest discovered causal links:")
disc["edges"].head(12)

Strongest discovered causal links:


,source,target,lag,strength,abs_strength
0,s_8,s_8,1,0.079120,0.079120
1,s_7,s_11,2,-0.075650,0.075650
2,s_15,s_11,1,0.069208,0.069208
3,s_11,s_4,2,0.067089,0.067089
4,s_12,s_2,2,-0.064091,0.064091
5,s_7,s_11,1,-0.063834,0.063834
6,s_11,s_21,1,-0.062225,0.062225
7,s_7,s_20,2,0.061412,0.061412
8,s_12,s_8,2,-0.060568,0.060568
9,s_4,s_3,0,0.060482,0.060482


In [5]:
import networkx as nx

G = nx.DiGraph()
for _, e in disc["edges"].iterrows():
    if e["source"] != e["target"]:
        G.add_edge(e["source"], e["target"], weight=e["abs_strength"])

fig, ax = plt.subplots(figsize=(10, 6.5))
pos = nx.spring_layout(G, seed=7, k=0.9)
widths = [2.6 * G[u][v]["weight"] / max(d["weight"] for *_, d in G.edges(data=True))
          for u, v in G.edges()]
nx.draw_networkx(G, pos, ax=ax, node_color="#dbeafe", edgecolors="#1f6feb",
                 node_size=1250, font_size=9, width=widths,
                 arrowsize=15, edge_color="#94a3b8")
ax.set_title(f"Causal graph discovered by PCMCI+ ({G.number_of_edges()} edges)")
ax.axis("off"); plt.tight_layout(); plt.show()

## Module 4 — Graph validation

### What "validation" can and cannot mean here

C-MAPSS ships **no ground-truth causal graph**. NASA released the simulator's outputs, not its
internal dependency structure. So it is impossible to check whether the discovered graph is
*correct*.

This is the single most important honesty point in the whole framework. Conflating "robust" with
"correct" is the most common overclaim in applied causal discovery papers, and an examiner will
look for it. What can be established is **robustness and plausibility**, via four checks:

1. **Stability** — re-run on random subsets of engines; do the same edges keep appearing?
2. **Sensitivity** — does the graph survive changes to the significance level and max lag?
3. **Cross-method** — does a completely different estimator agree on the strongest links?
4. **Domain plausibility** — do the edges respect the physical gas-path order of a turbofan?

(When this framework is later pointed at causRCA, which *does* have an expert ground-truth graph,
`validation.score_against_ground_truth` replaces all four with real precision/recall/SHD.)

In [6]:
if not cfg.RUN_VALIDATION:
    valid = None
    print("Validation skipped — set RUN_VALIDATION = True in pipeline_config.py to run it.")
else:
    valid = artifacts.cached("valid", lambda: validation.run(
        discovery_data, variables, disc["edges"],
        n_boot=cfg.BOOTSTRAP_RUNS,
        stability_threshold=cfg.STABILITY_THRESHOLD,
        tau_max=cfg.MAX_LAG,
        pc_alpha=cfg.SIGNIFICANCE,
        max_conds_dim=cfg.MAX_CONDITION_SET_SIZE,
        max_combinations=cfg.MAX_COMBINATIONS,
    ))
    print(f"Edges kept: {valid['n_validated']} of {len(disc['edges'])}")

Edges kept: 8 of 16


### Check 1 — Stability under bootstrap resampling

In [7]:
if valid is not None and not valid["stability"].empty:
    stab = valid["stability"]
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.hist(stab["recovery_rate"], bins=10, color="steelblue")
    ax.axvline(cfg.STABILITY_THRESHOLD, color="crimson", ls="--",
               label=f"keep threshold ({cfg.STABILITY_THRESHOLD})")
    ax.set_xlabel("fraction of bootstrap runs the edge reappeared in")
    ax.set_ylabel("number of edges")
    ax.set_title("Edge stability — edges on the right are trustworthy")
    ax.legend(fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()
    display(stab.head(10))
else:
    print("No stability results (validation not run).")

,source,target,lag,recovery_rate
0,s_7,s_11,2,1.000
1,s_7,s_11,1,0.875
2,s_8,s_8,1,0.750
3,s_11,s_21,1,0.625
4,s_15,s_11,1,0.625
5,s_12,s_2,2,0.500
6,s_11,s_4,2,0.500
7,s_7,s_20,2,0.500
8,s_2,s_12,1,0.375
9,s_4,s_11,2,0.375


### Check 2 — Sensitivity to the settings we chose

In [8]:
if valid is not None:
    print("If the graph changed completely whenever a knob moved, it would not be a finding.")
    display(valid["sensitivity"])

If the graph changed completely whenever a knob moved, it would not be a finding.


,pc_alpha,tau_max,n_edges,jaccard_vs_baseline
0,0.002,1,22,0.225806
1,0.002,2,4,0.250000
2,0.005,1,32,0.200000
3,0.005,2,9,0.388889
4,0.010,1,37,0.204545
5,0.010,2,16,1.000000


### Check 3 — Does an independent method agree?

In [9]:
if valid is not None:
    print("Top links according to lagged linear regression (a score-based method,")
    print("a different family from PCMCI+'s constraint-based approach):")
    display(valid["cross_method"].head(10))

Top links according to lagged linear regression (a score-based method,
a different family from PCMCI+'s constraint-based approach):


,source,target,lag,coef,abs_coef
0,s_12,s_2,2,-0.129785,0.129785
1,s_11,s_21,1,-0.126376,0.126376
2,s_12,s_8,2,-0.125603,0.125603
3,s_11,s_4,2,0.119931,0.119931
4,s_11,s_3,2,0.119195,0.119195
5,s_11,s_7,1,-0.111648,0.111648
6,s_7,s_20,2,0.110643,0.110643
7,s_12,s_4,2,-0.101948,0.101948
8,s_8,s_8,1,0.101906,0.101906
9,s_11,s_8,2,0.095247,0.095247


### Check 4 — Do the edges make physical sense?

Air flows through a turbofan in a fixed order: fan inlet → low-pressure compressor →
high-pressure compressor → combustor → high-pressure turbine → low-pressure turbine. An edge
running with that flow is physically expected. One running against it needs a feedback-control
explanation, so a high share of "downstream-consistent" edges is evidence the graph is picking up
real physics rather than noise.

In [10]:
if valid is not None and valid["domain"].get("n_checked"):
    dom = valid["domain"]
    print(f"Edges checked against gas-path order : {dom['n_checked']}")
    print(f"Consistent with physical flow        : {dom['n_downstream_consistent']}")
    print(f"Share                                : {dom['share_downstream_consistent']:.0%}")
    display(dom["detail"][["source", "target", "lag", "source_rank", "target_rank"]].head(10))

Edges checked against gas-path order : 16
Consistent with physical flow        : 9
Share                                : 56%


,source,target,lag,source_rank,target_rank
0,s_8,s_8,1,2,2
1,s_7,s_11,2,3,3
2,s_15,s_11,1,6,3
3,s_11,s_4,2,3,6
4,s_12,s_2,2,4,2
5,s_7,s_11,1,3,3
6,s_11,s_21,1,3,6
7,s_7,s_20,2,3,5
8,s_12,s_8,2,4,2
9,s_4,s_3,0,6,3


## Save for the next notebook

In [11]:
if valid is not None and not valid["validated_edges"].empty:
    edges = valid["validated_edges"]
    print(f"Passing {len(edges)} validated edges forward.")
else:
    edges = disc["edges"]
    print(f"Passing all {len(edges)} discovered edges forward (no validation filter).")

artifacts.save("edges_final", edges)
print("\nNext: 03_scm_rca_intervention_counterfactual")

Passing 8 validated edges forward.

Next: 03_scm_rca_intervention_counterfactual


## What this notebook established

| Question | Answer |
|---|---|
| Does PCMCI+ run on C-MAPSS? | Yes — the data format is ideal (regular, continuous, poolable) |
| Can the graph be scored for correctness? | **No** — C-MAPSS has no ground-truth graph |
| So what was validated? | Robustness: stability, sensitivity, cross-method agreement, physical plausibility |
| Any practical obstacle found? | Yes — discovery cost grows with graph density, so search bounds and variable reduction are required, not optional |